# 04 · Interactive data layer: cost calculator & experiment explorer

The site's two interactive pages — the **Cost Calculator** and the **Experiment
Explorer** — are pure client-side browsers over committed JSON files in
`website/data/`. This notebook is a **read-only** tour of that data layer: no API
calls, no model credits, nothing is written back. Every number shown here is exactly
what the interactive pages display in the browser.

Data files (`scripts/site/build_site.py` regenerates them from the committed markdown
reports):

- `experiments.json` — every recorded eval run.
- `per-class-accuracy.json` — per-class correctness per report.
- `cost-models.json` — measured per-model pricing + scale-up projections.
- `confusion-matrices.json` — confusion heatmap matrices per run.


## 0. Bootstrap: repo path

In [1]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


Repo root: /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone


## 1. Load the committed data layer

In [2]:
import json

data_dir = ROOT / "website" / "data"
experiments = json.loads((data_dir / "experiments.json").read_text())
per_class = json.loads((data_dir / "per-class-accuracy.json").read_text())
cost_models = json.loads((data_dir / "cost-models.json").read_text())
confusions = json.loads((data_dir / "confusion-matrices.json").read_text())

print("experiments.json      :", len(experiments["experiments"]), "runs")
print("per-class-accuracy.json:", len(per_class), "reports")
print("cost-models.json      :", len(cost_models["models"]), "models")
print("confusion-matrices.json:", len(confusions), "runs")


experiments.json      : 18 runs
per-class-accuracy.json: 14 reports
cost-models.json      : 4 models
confusion-matrices.json: 13 runs


## 2. Summary statistics (what the explorer's summary strip shows)

In [3]:
runs = experiments["experiments"]
models = {r["model_short"] for r in runs}
total_images = sum(r.get("images") or 0 for r in runs)
best = max(runs, key=lambda r: r.get("accuracy") or 0)

print(f"{len(runs)} runs | {len(models)} models | {total_images:,} images classified")
print(f"Best run: {best['model_short']} · {best['prompt_version']} · "
      f"{best['accuracy'] * 100:.1f}% ({best.get('correct')}/{best.get('total')}) "
      f"on {best.get('dataset', '')[:40]}")


18 runs | 5 models | 6,684 images classified
Best run: qwen3.7-flash · v11.8 · 98.7% (157/159) on fixed_size_sampled` (10 per class × 16 c


## 3. Top runs by exact-match accuracy

In [4]:
top = sorted(runs, key=lambda r: r.get("accuracy") or 0, reverse=True)[:8]
print(f"{'model':<28} {'prompt':<8} {'imgs':>5} {'acc%':>6}  {'correct':>7}")
print("-" * 64)
for r in top:
    print(f"{r['model_short']:<28} {r['prompt_version']:<8} {r.get('images', 0):>5} "
          f"{(r.get('accuracy') or 0) * 100:>6.1f}  {r.get('correct', 0):>7}")


model                        prompt    imgs   acc%  correct
----------------------------------------------------------------
qwen3.7-flash                v11.8      160   98.7      157
qwen3.7-flash                v11        160   98.7      156
qwen3.5-35b-a3b              v11.8      160   98.7      155
qwen3.7-flash                v11.7      160   98.1      156
qwen3.7-flash                v11        236   87.7      207
qwen3.7-flash                v11.8      320   87.1      277
gemini-2.5-flash-lite        v11.8      160   86.9      139
qwen3.7-flash                v14        160   85.0      136


## 4. Per-class accuracy for the 1,120-image run

The explorer's **per-class picker** is fed by `per-class-accuracy.json`, keyed by
report. This reproduces the chart for the largest held-out run

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

report_key = "qwen3.7-flash_v11.8_1600_balanced_1120_final"
classes = per_class[report_key]
items = sorted(classes.items(), key=lambda kv: kv[1]["accuracy"])

fig, ax = plt.subplots(figsize=(10, 6))
names = [c for c, _ in items]
vals = [v["accuracy"] * 100 for _, v in items]
colors = ["#2a9d8f" if v >= 90 else "#e9c46a" if v >= 70 else "#e76f51" for v in vals]
ax.barh(names, vals, color=colors)
ax.axvline(82.6, color="#264653", linestyle="--", lw=1)
ax.text(82.6, -0.8, " overall 82.6%", color="#264653", fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("Exact-match accuracy (%)")
ax.set_title("Per-class accuracy — qwen3.7-flash v11.8 on the 1,120-image slice")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


/var/folders/fv/7353h_0d5jzfv61r_mhb5_f40000gn/T/ipykernel_13838/3701583759.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Cost projections (what the cost calculator's sliders drive)

`cost-models.json` records each model's measured per-image billing and the linear
scale-up to 800 / 25,000 / 320,000 images. The cheapest model wins by an order of
magnitude.

In [6]:
print(f"{'model':<34} {'$/image':>9} {'800':>8} {'25,000':>9} {'320,000':>10}")
print("-" * 74)
for m in sorted(cost_models["models"], key=lambda m: m["actual_cost_per_image"]):
    p = m.get("projections", {})
    print(f"{m['model']:<34} {m['actual_cost_per_image']:>9.4f} "
          f"{p.get(800, 0):>8.2f} {p.get(25000, 0):>9.2f} {p.get(320000, 0):>10.2f}")


model                                $/image      800    25,000    320,000
--------------------------------------------------------------------------
nex-agi/nex-n2-pro                    0.0033     0.00      0.00       0.00
moonshotai/kimi-k3                    0.0039     0.00      0.00       0.00
x-ai/grok-4.5                         0.0060     0.00      0.00       0.00
anthropic/claude-fable-5              0.0534     0.00      0.00       0.00


## Recap

1. The interactive pages are thin browsers over committed JSON — no API access.
2. `experiments.json` powers the runs table, filters, and accuracy-by-model/prompt/size views.
3. `per-class-accuracy.json` + `confusion-matrices.json` power the per-class and confusion pickers.
4. `cost-models.json` drives the cost calculator's projections.

Reproduce any chart on the site from this data — every number traces back to a
committed markdown report in `docs/experiments/` and `reports/`.